In [3]:
import pip
import torch
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights
from torchinfo import summary
torch.manual_seed(0)

In [4]:
def compression_factor(depth, in_channels, latent_channels, spatial_dims):
    # each axis shrinks by axis_reduction (e.g. depth=2 -> axis_reduction=4)
    axis_reduction = 2 ** depth
    # spatial_dims axes shrink at the same time, so the element count compounds: axis_reduction**spatial_dims
    spatial_factor = axis_reduction ** spatial_dims
    channel_factor = in_channels / latent_channels
    return axis_reduction, spatial_factor, channel_factor, spatial_factor * channel_factor
 
 
def print_compression(name, depth, in_channels, latent_channels, spatial_dims):
    """Print the spatial vs channel breakdown; torchinfo.summary already covers shapes and params."""
    axis, spatial, channel, total = compression_factor(depth, in_channels, latent_channels, spatial_dims)
    print(f"{name} compression factor (depth={depth}): axis /{axis:.0f} -> spatial x{spatial:.2f} (elements) * channels x{channel:.2f} = x{total:.2f}")
 
 
def show_architecture(ae, x):
    """Print the full layer-by-layer architecture, input/output shapes and parameter counts."""
    _ = summary(ae, input_data=x, col_names=("input_size", "output_size", "num_params"), verbose=1, depth=3)


In [5]:
class DownBlock1D(nn.Module):     """MLP en... de Germán Alonso Pinedo Díaz"""

class DownBlock1D(nn.Module):
    """MLP encoder step: merge pairs of samples, halving the length."""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(2 * in_channels, out_channels),
            nn.ReLU(inplace=True),
            nn.LayerNorm(out_channels),
        )
 
    def forward(self, x):
        # x: [batch, channels, length]
        batch, channels, length = x.shape
        if length % 2 != 0:
            raise ValueError("DownBlock1D expects an even length.")
        x = x.transpose(1, 2)                    # [batch, length, channels]
        x = x.reshape(batch, length // 2, 2 * channels)
        x = self.block(x)                        # [batch, length // 2, out_channels]
        return x.transpose(1, 2)                 # [batch, out_channels, length // 2]
 
 
class UpBlock1D(nn.Module):
    """MLP decoder step: split each sample into two, doubling the length."""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.out_channels = out_channels
        self.proj = nn.Linear(in_channels, 2 * out_channels)
        self.act = nn.ReLU(inplace=True)
        self.norm = nn.LayerNorm(out_channels)
 
    def forward(self, x):
        # x: [batch, channels, length]
        batch, _, length = x.shape
        x = x.transpose(1, 2)                    # [batch, length, channels]
        x = self.act(self.proj(x))               # [batch, length, 2 * out_channels]
        x = x.reshape(batch, length * 2, self.out_channels)
        x = self.norm(x)
        return x.transpose(1, 2)                 # [batch, out_channels, length * 2]

In [6]:
class Autoencoder1D(nn.Module):
    """Stacks `depth` DownBlock/UpBlock pairs."""
    def __init__(self, in_channels, latent_channels, depth=1):
        super().__init__()
        self.depth = depth
        channels = [in_channels] + [latent_channels] * depth
        self.encoder = nn.ModuleList(
            [DownBlock1D(channels[i], channels[i + 1]) for i in range(depth)]
        )

        # depth 2
        self.decoder = nn.ModuleList(
            [UpBlock1D(channels[i + 1], channels[i]) for i in reversed(range(depth))]
        )
 
    def encode(self, x):
        z = x
        for down in self.encoder:
            z = down(z)
        return z
 
    def decode(self, z):
        y = z
        for up in self.decoder:
            y = up(y)
        return y
 
    def forward(self, x):
        return self.decode(self.encode(x))
 
 
x1 = torch.randn(4, 3, 16)
 
ae1 = Autoencoder1D(in_channels=3, latent_channels=2, depth=1)
show_architecture(ae1, x1)
print_compression("1D", depth=1, in_channels=3, latent_channels=2, spatial_dims=1)
 
ae1_deep = Autoencoder1D(in_channels=3, latent_channels=2, depth=2)
show_architecture(ae1_deep, x1)
print_compression("1D (depth=2)", depth=2, in_channels=3, latent_channels=2, spatial_dims=1)

Layer (type:depth-idx)                   Input Shape               Output Shape              Param #
Autoencoder1D                            [4, 3, 16]                [4, 3, 16]                --
├─ModuleList: 1-1                        --                        --                        --
│    └─DownBlock1D: 2-1                  [4, 3, 16]                [4, 2, 8]                 --
│    │    └─Sequential: 3-1              [4, 8, 6]                 [4, 8, 2]                 18
├─ModuleList: 1-2                        --                        --                        --
│    └─UpBlock1D: 2-2                    [4, 2, 8]                 [4, 3, 16]                --
│    │    └─Linear: 3-2                  [4, 8, 2]                 [4, 8, 6]                 18
│    │    └─ReLU: 3-3                    [4, 8, 6]                 [4, 8, 6]                 --
│    │    └─LayerNorm: 3-4               [4, 16, 3]                [4, 16, 3]                6
Total params: 42
Trainable params: 4